In [137]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet 
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from plotnine import *
import warnings
from sklearn.exceptions import ConvergenceWarning

#suppress convergence warnings
warnings.simplefilter("ignore", ConvergenceWarning)
pd.options.mode.chained_assignment = None

In [139]:
train_data = pd.read_csv("train_new.csv")
train_data = train_data.dropna()
test_data = pd.read_csv("test_new.csv")

In [140]:
X = train_data.drop(["PID", "SalePrice"], axis = 1)
y = np.log(train_data["SalePrice"])

# write function that tunes regression

def tune_regression(X, y, model_type = "linear", alpha_values = [0.001, 0.01, 0.1, 1, 10], l1_ratio_values = [0.0, 0.25, 0.5, 0.75, 1.0], cv = 5):

    # create criteria for model specification
    if model_type  == "linear":
        model = LinearRegression()
        alpha = {}
        l1_ratio = {}
    elif model_type == "ridge":
        model = Ridge()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "lasso":
        model = Lasso()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "elasticnet":
        model = ElasticNet()
        alpha = {"regression__alpha": alpha_values, "regression__l1_ratio": l1_ratio_values}
    else:
        raise ValueError("Unsupported model_type. Choose from 'lasso', 'ridge', 'elasticnet', or 'linear'.")

    # create pipeline
    ct = ColumnTransformer([("dummify", OneHotEncoder(sparse_output = False, handle_unknown='ignore'),
                                        make_column_selector(dtype_include=object)),
                            ("standardize", StandardScaler(), 
                                        make_column_selector(dtype_include=np.number))],
                                        remainder = "passthrough")

    pipeline = Pipeline([
        ("preprocessing", ct), 
        ("regression", model)
    ])
    
    # do grid search
    grid_search = GridSearchCV(pipeline, alpha, cv=cv, scoring='neg_root_mean_squared_error')
    grid_search_fitted = grid_search.fit(X, y)

    # get best model
    best_model = grid_search.best_estimator_
    best_model_fitted = best_model.fit(X, y)

    # get coefs and var names
    coefs = best_model.named_steps['regression'].coef_
    feature_names = best_model_fitted.named_steps['preprocessing'].get_feature_names_out()

    # create df to store coefs
    coefs_df = pd.DataFrame({
        "Feature Name": feature_names,
        "Coefficients": coefs})
    # mse = -grid_search_fitted.cv_results_['mean_test_score']
    # rmse = np.sqrt(mse)

    print("Cross-validated MSE scores:", -grid_search_fitted.cv_results_['mean_test_score'])

    # extract best model values
    best_alpha = grid_search.best_params_.get("regression__alpha", None)
    best_l1_ratio = grid_search.best_params_.get("regression__l1_ratio", None)
    best_score = grid_search.best_score_
    
    # print best alpha and l1_ratio scores if applicable
    if best_alpha is not None:
        print(f"Best alpha: {best_alpha}")
    if best_l1_ratio is not None:
        print(f"Best l1 ratio: {best_l1_ratio}")
    print(f"Best cross-validated MSE score: {-best_score}")
    sorted_coefs_df = coefs_df.sort_values(by='Coefficients', key = abs, ascending=False)

    
    return best_model_fitted

In [149]:

def tune_regression(X, y, model_type = "linear", alpha_values=np.linspace(0.00, 1.0, 15), l1_ratio_values=np.linspace(0.00, 1.0, 15), cv = 5, test_size = 0.2):

    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    # create criteria for model specification
    if model_type  == "linear":
        model = LinearRegression()
        alpha = {}
        l1_ratio = {}
    elif model_type == "ridge":
        model = Ridge()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "lasso":
        model = Lasso()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "elasticnet":
        model = ElasticNet()
        alpha = {"regression__alpha": alpha_values, "regression__l1_ratio": l1_ratio_values}
    else:
        raise ValueError("Unsupported model_type. Choose from 'lasso', 'ridge', 'elasticnet', or 'linear'.")

    # create pipeline
    ct = ColumnTransformer([("dummify", OneHotEncoder(sparse_output = False, handle_unknown='ignore'),
                                        make_column_selector(dtype_include=object)),
                            ("standardize", StandardScaler(), 
                                        make_column_selector(dtype_include=np.number))],
                                        remainder = "passthrough")

    pipeline = Pipeline([
        ("preprocessing", ct), 
        ("regression", model)
    ])
    
    # do grid search
    grid_search = GridSearchCV(pipeline, alpha, cv=cv, scoring='neg_root_mean_squared_error')
    grid_search_fitted = grid_search.fit(X_train, y_train)

    # get best model
    best_model = grid_search.best_estimator_
    best_model_fitted = best_model.fit(X, y)

    # Predict on the test set
    y_pred = best_model_fitted.predict(X_test)
    print(f"Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

    # Print best hyperparameters
    best_alpha = grid_search.best_params_.get("regression__alpha", None)
    best_l1_ratio = grid_search.best_params_.get("regression__l1_ratio", None)
    best_score = grid_search.best_score_

    if best_alpha is not None:
        print(f"Best alpha: {best_alpha}")
    if best_l1_ratio is not None:
        print(f"Best l1 ratio: {best_l1_ratio}")
    print(f"Best cross-validated RMSE score: {-best_score}")

    return best_model_fitted


In [150]:
tune_regression(X, y, model_type="ridge")
tune_regression(X, y, model_type="lasso")
tune_regression(X, y, model_type="elasticnet")

Test RMSE: 0.13087884978772085
Best alpha: 1.0
Best cross-validated RMSE score: 0.15938228918464764
Test RMSE: 0.13087884978772085
Best alpha: 1.0
Best cross-validated RMSE score: 0.15938228918464764


Test RMSE: 0.1375144955198189
Best alpha: 0.0
Best cross-validated RMSE score: 0.15270126058161052
Test RMSE: 0.1375144955198189
Best alpha: 0.0
Best cross-validated RMSE score: 0.15270126058161052


Test RMSE: 0.14912963078600236
Best alpha: 0.0
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1517857288172934
Test RMSE: 0.14912963078600236
Best alpha: 0.0
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1517857288172934


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f9ba86a9a20>),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f9ba86aa8c0>)])),
                ('regression',
                 ElasticNet(alpha=np.float64(0.0), l1_ratio=np.float64(0.0)))])

In [156]:
final_model_fit1 = tune_regression(X, y, model_type="elasticnet")


Test RMSE: 0.16474911095399497
Best alpha: 0.0
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1430904292456133
Test RMSE: 0.16474911095399497
Best alpha: 0.0
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1430904292456133


In [160]:
final_predictions_regression1 = pd.DataFrame(
    {"PID": test_data['PID'],
    "SalePrice": np.exp(final_model_fit1.predict(test_data))}
)

final_predictions_regression1.to_csv('final_predictions_regression1.csv', index = False)

In [168]:

def tune_regression(X, y, model_type = "linear", alpha_values=[0.0, 0.01, 0.1, 0.11, 0.5, 0.75, 1.0], l1_ratio_values=[0.0, 0.01, 0.1, 0.11, 0.5, 0.75, 1.0], cv = 5, test_size = 0.2):

    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    # create criteria for model specification
    if model_type  == "linear":
        model = LinearRegression()
        alpha = {}
        l1_ratio = {}
    elif model_type == "ridge":
        model = Ridge()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "lasso":
        model = Lasso()
        alpha = {"regression__alpha": alpha_values}
        l1_ratio = {}
    elif model_type == "elasticnet":
        model = ElasticNet()
        alpha = {"regression__alpha": alpha_values, "regression__l1_ratio": l1_ratio_values}
    else:
        raise ValueError("Unsupported model_type. Choose from 'lasso', 'ridge', 'elasticnet', or 'linear'.")

    # create pipeline
    ct = ColumnTransformer([("dummify", OneHotEncoder(sparse_output = False, handle_unknown='ignore'),
                                        make_column_selector(dtype_include=object)),
                            ("standardize", StandardScaler(), 
                                        make_column_selector(dtype_include=np.number))],
                                        remainder = "passthrough")

    pipeline = Pipeline([
        ("preprocessing", ct), 
        ("regression", model)
    ])
    
    # do grid search
    grid_search = GridSearchCV(pipeline, alpha, cv=cv, scoring='neg_root_mean_squared_error')
    grid_search_fitted = grid_search.fit(X_train, y_train)

    # get best model
    best_model = grid_search.best_estimator_
    best_model_fitted = best_model.fit(X, y)

    # Predict on the test set
    y_pred = best_model_fitted.predict(X_test)
    print(f"Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

    # Print best hyperparameters
    best_alpha = grid_search.best_params_.get("regression__alpha", None)
    best_l1_ratio = grid_search.best_params_.get("regression__l1_ratio", None)
    best_score = grid_search.best_score_

    if best_alpha is not None:
        print(f"Best alpha: {best_alpha}")
    if best_l1_ratio is not None:
        print(f"Best l1 ratio: {best_l1_ratio}")
    print(f"Best cross-validated RMSE score: {-best_score}")

    return best_model_fitted


In [169]:
final_model_fit2 = tune_regression(X, y, model_type="elasticnet")

Test RMSE: 0.17647140760403374
Best alpha: 0.01
Best l1 ratio: 0.01
Best cross-validated RMSE score: 0.14619075492719685
Test RMSE: 0.17647140760403374
Best alpha: 0.01
Best l1 ratio: 0.01
Best cross-validated RMSE score: 0.14619075492719685


In [170]:
final_predictions_regression2 = pd.DataFrame(
    {"PID": test_data['PID'],
    "SalePrice": np.exp(final_model_fit2.predict(test_data))}
)

final_predictions_regression2.to_csv('final_predictions_regression2.csv', index = False)

In [176]:
final_model_fit3 = tune_regression(X, y, model_type="elasticnet")

Test RMSE: 0.18740947014870146
Best alpha: 0.01
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1412943246324133
Test RMSE: 0.18740947014870146
Best alpha: 0.01
Best l1 ratio: 0.0
Best cross-validated RMSE score: 0.1412943246324133


In [177]:
final_predictions_regression3 = pd.DataFrame(
    {"PID": test_data['PID'],
    "SalePrice": np.exp(final_model_fit3.predict(test_data))}
)

final_predictions_regression3.to_csv('final_predictions_regression3.csv', index = False)